# 1º Script [Python] - Extração dos livros mais lidos do Skoob
### Lucas Tavares Brito
### 12/08/2023

Fiz esse script no fim do primeiro semestre, para um pré-projeto de pesquisa na área de economia da cultura.

Nele, utilizei as bibliotecas "Requests", "BeautifulSoup" e "Pandas" para fazer a extração da lista de links dos livros mais lidos da rede social de livros Skoob. Em seguida, eu extraio os dados de cada um dos livros mais vendidos presentes em suas páginas:
- título;
- autor;
- gênero;
- quantidade de pessoas que leram;
- quantidade de pessoas que estão lendo;
- quantidade de pessoas que querem ler;
- quantidade de pessoas que estão relendo;
- quantidade de pessoas que abandonaram o livro;
- quantidade de resenhas sobre o livro;
- quantidade de avaliações;
- percentual de pessoas que avaliaram o livro com 1 estrela;
- percentual de pessoas que avaliaram o livro com 2 estrelas;
- percentual de pessoas que avaliaram o livro com 3 estrelas;
- percentual de pessoas que avaliaram o livro com 4 estrelas;
- percentual de pessoas que avaliaram o livro com 5 estrelas;
- percentual de leitores do sexo masculino;
- percentual de leitores do sexo feminino.


Após salvar todas essas informações em um dataframe, exporto os dados em formato CSV.

No 2º script, faço análises com os dados utilizando o R.

In [ ]:
# Importar bibliotecas necessárias para a operação
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

import datetime
import pytz

In [ ]:
# Obter data e hora locais, para rotular os arquivos
dataHora = datetime.datetime.now(pytz.timezone('Brazil/East')).strftime("%Y%m%d-%H%M%S")

## A primeira extração

Aqui, nós queremos extrair as URLs e o número de leituras de cada livro dessa página (os mais lidos do Skoob): https://www.skoob.com.br/livro/top_mais/lidos/

In [ ]:
# Criar a lista onde serão colocados os links dos livros mais lidos
maisLidos = []

# Requisitar html do site
url = 'https://www.skoob.com.br/livro/top_mais/lidos/'
pagina = requests.get(url)
conteudo = pagina.content

# Transformar página em arquivo do BeautifulSoup e pegar dados de cada um dos livros
site = BeautifulSoup(conteudo, 'html.parser')
livros = site.find_all('div', attrs={'class': 'livro-capa-mini'})
for livro in livros:
    link = 'https://www.skoob.com.br'+livro.find('a').get('href') # Pegar link
    numero = livro.find('div', attrs={'style': 'border:0px solid red; margin-top:0px;'}).text # Pegar total de leituras
    maisLidos.append([link, numero]) # Adicionar ao final da lista

# Transformar a lista em um dataframe do Pandas
maisLidos = pd.DataFrame(maisLidos, columns = ['link','numeroDeLeituras'])

# Jogar para csv "maisLidos_skoob"
caminhoCSV_maisLidos = './'+dataHora+'_maisLidos_skoob.csv'
maisLidos.to_csv(caminhoCSV_maisLidos, index = False)

In [ ]:
# Consultar primeiras linhas do dataframe
maisLidos.head()

,link,numeroDeLeituras
0,https://www.skoob.com.br/o-pequeno-principe-69...,789.863
1,https://www.skoob.com.br/harry-potter-e-a-pedr...,743.705
2,https://www.skoob.com.br/a-culpa-e-das-estrela...,628.527
3,https://www.skoob.com.br/harry-potter-e-a-cama...,628.159
4,https://www.skoob.com.br/harry-potter-e-o-pris...,577.593


## A segunda extração

Agora, com os links em mãos (formato CSV), queremos extrair todas as informações de cada um dos livros. As páginas dos livros seguem esse formato: https://www.skoob.com.br/o-pequeno-principe-693ed56597.html

In [ ]:
# Criar lista vazia para inserção dos dados dos livros
dados_livros = []

# Importar dados do arquivo "data-hora_maisLidos_skoob", da pasta "1º Script - Arquivos de Input e Output"
lista_livros = pd.read_csv(caminhoCSV_maisLidos)
lista_livros = pd.DataFrame(lista_livros)

# Criar loop que acessa cada uma das URLs fornecidas
for link in lista_livros['link']:
    # Pegar conteúdo da página
    pagina = requests.get(link)
    conteudo = pagina.content
    site = BeautifulSoup(conteudo, 'html.parser')

    # Pegar informações do livro
    titulo = site.find('strong', attrs={'class': 'sidebar-titulo'}).text

    # Pegar gênero, se esse existir
    genero = site.find('span', attrs={'class': 'pg-livro-generos'})
    if genero is None:
        genero = 'NA'
    else:
        genero = genero.text

    # Pegar autor
    autor = site.find('div', attrs={'id': 'pg-livro-menu-principal-container'}).find_all('a')[1].text

    local_autor = site.find('div', attrs={'id': 'pg-livro-menu-principal-container'})
    local_autor = 'https://www.skoob.com.br'+local_autor.find_all('a')[1].get('href')
    local_autor = requests.get(local_autor).content
    local_autor = BeautifulSoup(local_autor, 'html.parser')
    local_autor = local_autor.find('span', attrs={'class': 'adr'}).text

    # Pegar status
    status = site.find('div', attrs={'id': 'livro-perfil-status'})
    status_leram = status.find_all('div', attrs={'class': 'bar'})[-1].find('b').find('a').text
    status_lendo = status.find_all('div', attrs={'class': 'bar'})[-2].find('b').find('a').text
    status_queremLer = status.find_all('div', attrs={'class': 'bar'})[-3].find('b').find('a').text
    status_relendo = status.find_all('div', attrs={'class': 'bar'})[-4].find('b').find('a').text
    status_abandonos = status.find_all('div', attrs={'class': 'bar'})[-5].find('b').find('a').text
    status_resenhas = status.find_all('div', attrs={'class': 'bar'})[-6].find('b').find('a').text

    # Pegar média de avaliações
    media_avaliacoes = site.find('span', attrs={'class': 'rating'}).text

    # Pegar número de avaliações
    num_avaliacoes = site.find('div', attrs={'id': 'pg-livro-box-rating-avaliadores-numero'}).text

    # Pegar dados das avaliações
    dadosAvaliacoes = site.find('div', attrs={'class': 'pg-livro-box-avaliacoes-grafico'}).find_all('div', attrs={'style': 'height:12px; margin-bottom:4px; float:left;'})

    percent_1estrela = dadosAvaliacoes[-1].text
    percent_2estrela = dadosAvaliacoes[-2].text
    percent_3estrela = dadosAvaliacoes[-3].text
    percent_4estrela = dadosAvaliacoes[-4].text
    percent_5estrela = dadosAvaliacoes[-5].text

    # Pegar percentual masculino e feminino
    percent_masc = site.find('span', attrs={'class': 'pg-livro-icone-male-label'}).text
    percent_femi = site.find('span', attrs={'class': 'pg-livro-icone-female-label'}).text

    # Pegar tags populares, se essas existirem
    tags = site.find('ul', attrs={'id': 'tags'})
    if tags is None:
        tags_populares = ''
    else:
        tags = tags.find_all('li', attrs={'class': 'litags'})
        tags_populares = ''
        for tag in tags:
            tags_populares = tags_populares+', '+tag.text
        tags_populares = tags_populares[2:]

    # Pegar site das resenhas, se esse existir
    caminho_resenha = site.find('div', attrs={'id': 'livro-perfil-resenhas'})
    if caminho_resenha is None:
        link_resenhas = 'NA'
    else:
        link_resenhas = 'https://www.skoob.com.br'+caminho_resenha.find('a').get('href')
        match = re.search(r'/resenhas/(\d+)/edicao', link_resenhas)
        id_livro = match.group(1)

    # Jogar tudo na lista inicial
    dados_livros.append([id_livro, titulo, genero, autor, local_autor, status_leram, status_lendo, status_queremLer, status_relendo, status_abandonos, status_resenhas, media_avaliacoes, num_avaliacoes, percent_1estrela, percent_2estrela, percent_3estrela, percent_4estrela, percent_5estrela, percent_masc, percent_femi, tags_populares, link_resenhas])

# Transformar a lista em um dataframe do Pandas
dataFrame_livros = pd.DataFrame(dados_livros, columns = ['id_livro', 'titulo', 'genero', 'autor', 'local_autor', 'status_leram', 'status_lendo', 'status_queremLer', 'status_relendo', 'status_abandonos', 'status_resenhas', 'media_avaliacoes', 'num_avaliacoes', 'percent_1estrela', 'percent_2estrela', 'percent_3estrela', 'percent_4estrela', 'percent_5estrela', 'percent_masc', 'percent_femi', 'tags_populares', 'link_resenhas'])

# Jogar para csv "data-hora_informacoesLivros_skoob"
caminhoCSV_informacoesLivros = './'+dataHora+'_informacoesLivros_skoob.csv'
dataFrame_livros.to_csv(caminhoCSV_informacoesLivros, index = False)

In [ ]:
# Consultar primeiras linhas do dataframe
dataFrame_livros.head()

,id_livro,titulo,genero,autor,local_autor,status_leram,status_lendo,status_queremLer,status_relendo,status_abandonos,...,num_avaliacoes,percent_1estrela,percent_2estrela,percent_3estrela,percent_4estrela,percent_5estrela,percent_masc,percent_femi,tags_populares,link_resenhas
0,693,O Pequeno Príncipe,Infantil / Ficção / Literatura Estrangeira,Antoine de Saint-Exupéry,\nFrança \n- Lyon \n,789.936,11.597,181.339,2.174,3.945,...,244.936 avaliações,1%,2%,9%,21%,67%,16%,84%,"literatura infanto-juvenil, filosofia, aventur...",https://www.skoob.com.br/livro/resenhas/693/ed...
1,108,Harry Potter e a Pedra Filosofal,Aventura / Fantasia / Literatura Estrangeira /...,J.K. Rowling,\nInglaterra \n- Gloucestershire - Yate \n,743.705,19.924,168.110,2.732,4.546,...,281.490 avaliações,0%,1%,8%,24%,67%,21%,79%,"j.k.rowling, j.k. rowling, literatura inglesa,...",https://www.skoob.com.br/livro/resenhas/108/ed...
2,247555,A Culpa é das Estrelas,Literatura Estrangeira / Romance,John Green,\nEstados Unidos \n- Indiana - Indianápolis \n,628.527,13.452,277.696,1.190,6.549,...,193.045 avaliações,1%,4%,15%,29%,51%,10%,90%,,https://www.skoob.com.br/livro/resenhas/247555...
3,357,Harry Potter e a Câmara Secreta,Aventura / Fantasia / Ficção / Literatura Estr...,J.K. Rowling,\nInglaterra \n- Gloucestershire - Yate \n,628.159,12.004,133.536,1.238,2.610,...,228.241 avaliações,0%,1%,9%,26%,63%,21%,79%,"literatura infanto-juvenil, literatura inglesa...",https://www.skoob.com.br/livro/resenhas/357/ed...
4,358,Harry Potter e o Prisioneiro de Azkaban,Aventura / Fantasia / Literatura Estrangeira /...,J.K. Rowling,\nInglaterra \n- Gloucestershire - Yate \n,577.592,9.766,159.002,1.017,1.984,...,205.455 avaliações,0%,0%,4%,16%,78%,21%,79%,"bruxo, bruxaria, literatura infanto-juvenil, l...",https://www.skoob.com.br/livro/resenhas/358/ed...
